# EDA, Hypothesis Testing, and Machine Learning Preparation

**Dataset:** Wisconsin Diagnostic Breast Cancer (WDBC)  
**Goal:** Explore the data, test hypotheses, prepare features, and evaluate a baseline classifier.

> Educational demonstration only. This model is not a clinical diagnostic tool.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay

data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target  # 0 = malignant, 1 = benign
df = X.copy()
df["diagnosis"] = y.map({0: "malignant", 1: "benign"})
df.head()

## 1. Dataset summary and quality checks

In [ ]:
print("Shape:", df.shape)
print("\\nData types:\\n", df.dtypes.value_counts())
print("\\nMissing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\\nClass counts:\\n", df["diagnosis"].value_counts())
display(df.describe().T)

## 2. Target distribution

In [ ]:
df["diagnosis"].value_counts().plot(kind="bar", title="Diagnosis Class Distribution")
plt.xlabel("Diagnosis"); plt.ylabel("Count"); plt.xticks(rotation=0); plt.show()

## 3. Feature comparison

In [ ]:
df.boxplot(column="mean radius", by="diagnosis")
plt.title("Mean Radius by Diagnosis"); plt.suptitle(""); plt.ylabel("Mean radius"); plt.show()

## 4. Correlation analysis

In [ ]:
selected = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness", "mean compactness", "mean concavity", "mean concave points", "mean symmetry", "mean fractal dimension"]
corr = X[selected].corr()
plt.figure(figsize=(9, 7))
plt.imshow(corr, vmin=-1, vmax=1, aspect="auto")
plt.xticks(range(len(selected)), [s.replace("mean ", "") for s in selected], rotation=60, ha="right")
plt.yticks(range(len(selected)), [s.replace("mean ", "") for s in selected])
plt.colorbar(label="Pearson correlation"); plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()

## 5. Hypotheses

- **H1 (primary):** Mean-radius distributions differ between malignant and benign cases.
- **H2:** Diagnosis is associated with whether mean radius is above the dataset median.
- **H3:** Mean concave-points distributions differ between malignant and benign cases.

We use a two-sided Mann–Whitney U test for H1 because it compares two independent groups without assuming normality.

In [ ]:
malignant = df[df["diagnosis"] == "malignant"]
benign = df[df["diagnosis"] == "benign"]
u_stat, p_value = mannwhitneyu(malignant["mean radius"], benign["mean radius"], alternative="two-sided")
print(f"Mann–Whitney U = {u_stat:.1f}; p-value = {p_value:.3e}")
print("Decision at alpha=0.05:", "Reject H0" if p_value < 0.05 else "Fail to reject H0")

In [ ]:
median_radius = X["mean radius"].median()
radius_group = pd.cut(X["mean radius"], bins=[-np.inf, median_radius, np.inf], labels=["at/below median", "above median"])
table = pd.crosstab(df["diagnosis"], radius_group)
chi2, p_chi, dof, expected = chi2_contingency(table)
display(table)
print(f"Chi-square = {chi2:.2f}; df = {dof}; p-value = {p_chi:.3e}")

## 6. Data preparation and baseline model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=42))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Precision (benign):", round(precision_score(y_test, pred, pos_label=1), 3))
print("Recall (benign):", round(recall_score(y_test, pred, pos_label=1), 3))
print("F1 (benign):", round(f1_score(y_test, pred, pos_label=1), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["Malignant", "Benign"], values_format="d")
plt.title("Baseline Logistic Regression — Test Set"); plt.show()

## 7. Conclusion and next steps

The data contains useful signal for distinguishing the two diagnosis labels. Mean radius differs significantly between groups, and several geometric features are correlated. Next steps include effect sizes and confidence intervals, cross-validation, comparing models, checking calibration, and external validation. Do not use this educational baseline for clinical decisions.